# Sutro · Bangla handwritten digit classifier (NumtaDB, small CNN)

Classifies one isolated handwritten Bangla digit (০–৯). Input: 32×32 grayscale, ink bright (the preprocessing stored in `numta_32x32.npz`, produced by `tools/convert_numta.py` from the Kaggle dataset `BengaliAI/numta`, CC BY-SA 4.0).

**Scope.** This is an *input-assistance* model for single digits only. It is **not** a khatian reader and says nothing about land verification. Split: the npz holds a seeded 90/10 train/validation split of NumtaDB's training sets a–e (grouping by writer is not available in NumtaDB's public metadata, so the split is by image — disclosed in the model card). A separate held-out **test** slice is carved from the validation half here so that model selection (validation) and the final number (test) use different images.

In [ ]:
import os, json, time, random, glob, collections
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = int(os.environ.get("EPOCHS", 2 if SMOKE else 20)); BATCH = 256; LR = 2e-3
RUN_ID = time.strftime("numta-cnn-%Y%m%d-%H%M%S"); OUT = os.path.join(os.environ.get("RUN_DIR", "run"), RUN_ID); os.makedirs(OUT, exist_ok=True)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| run:", RUN_ID)

In [ ]:
cands = glob.glob("/kaggle/input/**/numta_32x32.npz", recursive=True) + [os.environ.get("NUMTA_NPZ", "C:/land/sutro-dataset/real_public/numta_digits/numta_32x32.npz")]
NPZ = next(p for p in cands if os.path.exists(p)); print("data:", NPZ)
d = np.load(NPZ)
x_tr, y_tr, d_tr = d["x_train"], d["y_train"].astype(np.int64), d["d_train"]
x_va_all, y_va_all, d_va_all = d["x_val"], d["y_val"].astype(np.int64), d["d_val"]
# carve a held-out test half out of the validation slice (seeded), so val != test
idx = np.random.RandomState(SEED).permutation(len(x_va_all)); h = len(idx) // 2
x_va, y_va, d_va = x_va_all[idx[:h]], y_va_all[idx[:h]], d_va_all[idx[:h]]
x_te, y_te, d_te = x_va_all[idx[h:]], y_va_all[idx[h:]], d_va_all[idx[h:]]
if SMOKE: x_tr, y_tr, d_tr = x_tr[:4000], y_tr[:4000], d_tr[:4000]
print("train", x_tr.shape, "val", x_va.shape, "test", x_te.shape)
print("class counts (train):", dict(collections.Counter(y_tr.tolist())))
print("source sets a..e (train):", dict(collections.Counter(d_tr.tolist())))
import hashlib; data_sha = hashlib.sha256(open(NPZ, "rb").read()).hexdigest()
json.dump(dict(source="Kaggle BengaliAI/numta (NumtaDB), CC BY-SA 4.0", file=os.path.basename(NPZ), sha256=data_sha, preprocessing="grayscale, 32x32 bilinear, inverted (ink bright)",
               n_train=int(len(x_tr)), n_val=int(len(x_va)), n_test=int(len(x_te)), grouping="by image (writer ids not available); val/test carved from NumtaDB train sets a-e",
               class_counts_train={int(k): int(v) for k, v in collections.Counter(y_tr.tolist()).items()}), open(os.path.join(OUT, "dataset_manifest.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
json.dump(dict(seed=SEED, val_indices_of_npz_val=idx[:h].tolist(), test_indices_of_npz_val=idx[h:].tolist()), open(os.path.join(OUT, "split_manifest.json"), "w"))

In [ ]:
def tens(x): return torch.from_numpy(x.astype(np.float32) / 255.0).unsqueeze(1)
class DS(torch.utils.data.Dataset):
    def __init__(s, x, y, aug): s.x, s.y, s.aug = tens(x), torch.from_numpy(y), aug
    def __len__(s): return len(s.x)
    def __getitem__(s, i):
        x = s.x[i]
        if s.aug:   # small random affine + noise
            th = torch.zeros(1, 2, 3); a = random.uniform(-.15, .15); sc = random.uniform(.9, 1.1)
            th[0, 0, 0] = sc * np.cos(a); th[0, 0, 1] = -sc * np.sin(a); th[0, 1, 0] = sc * np.sin(a); th[0, 1, 1] = sc * np.cos(a)
            th[0, 0, 2] = random.uniform(-.1, .1); th[0, 1, 2] = random.uniform(-.1, .1)
            g = F.affine_grid(th, (1, 1, 32, 32), align_corners=False); x = F.grid_sample(x[None], g, align_corners=False, padding_mode="zeros")[0]
            x = (x + torch.randn_like(x) * random.uniform(0, .05)).clamp(0, 1)
        return x, s.y[i]
mk = lambda x, y, aug, sh: torch.utils.data.DataLoader(DS(x, y, aug), batch_size=BATCH, shuffle=sh, num_workers=0)
dl_tr, dl_va, dl_te = mk(x_tr, y_tr, True, True), mk(x_va, y_va, False, False), mk(x_te, y_te, False, False)

def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2), nn.Dropout(.2))
model = nn.Sequential(blk(1, 32), blk(32, 64), blk(64, 128), nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True), nn.Dropout(.4), nn.Linear(256, 10)).to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(dl_tr))
json.dump(dict(run_id=RUN_ID, task="isolated Bangla digit classification (10 classes)", model="2x[conv-bn-relu]x3 + MLP, 32x32 gray", epochs=EPOCHS, batch=BATCH, lr=LR, optimizer="AdamW+OneCycle",
               augmentation="random affine (+-0.15 rad, 0.9-1.1 scale, +-10% shift) + gaussian noise", seed=SEED, device=DEVICE, framework=f"torch {torch.__version__}"), open(os.path.join(OUT, "config.json"), "w"), indent=1)

In [ ]:
@torch.no_grad()
def evaluate(dl):
    model.eval(); P, Y = [], []
    for x, y in dl: P.append(model(x.to(DEVICE)).argmax(1).cpu()); Y.append(y)
    P, Y = torch.cat(P).numpy(), torch.cat(Y).numpy()
    cm = np.zeros((10, 10), np.int64)
    for t, p in zip(Y, P): cm[t, p] += 1
    tp = np.diag(cm); prec = tp / np.maximum(cm.sum(0), 1); rec = tp / np.maximum(cm.sum(1), 1); f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-9)
    return dict(accuracy=float(tp.sum() / cm.sum()), macro_f1=float(f1.mean()), per_class=[dict(cls=int(k), precision=float(prec[k]), recall=float(rec[k]), f1=float(f1[k]), n=int(cm[k].sum())) for k in range(10)]), cm

history = []; best = -1; t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model.train(); tot = n = 0
    for x, y in dl_tr:
        x, y = x.to(DEVICE), y.to(DEVICE); loss = F.cross_entropy(model(x), y, label_smoothing=.05)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step(); tot += loss.item() * len(y); n += len(y)
    m, _ = evaluate(dl_va); history.append(dict(epoch=ep, train_loss=tot / n, val_accuracy=m["accuracy"], val_macro_f1=m["macro_f1"]))
    flag = ""
    if m["accuracy"] > best: best = m["accuracy"]; torch.save(model.state_dict(), os.path.join(OUT, "best.pt")); flag = " *"
    print(f"epoch {ep:2d}/{EPOCHS} loss {tot / n:.4f} val acc {m['accuracy']:.4f} macro-F1 {m['macro_f1']:.4f} ({time.time() - t0:.0f}s){flag}")
json.dump(history, open(os.path.join(OUT, "history.json"), "w"), indent=1)

In [ ]:
model.load_state_dict(torch.load(os.path.join(OUT, "best.pt"), map_location=DEVICE))
test, cm = evaluate(dl_te); val, _ = evaluate(dl_va)
# per source-set accuracy on test (NumtaDB sets differ in capture conditions)
model.eval(); P = torch.cat([model(x.to(DEVICE)).argmax(1).cpu() for x, _ in dl_te]).numpy()
by_src = {f"set_{'abcde'[s]}": dict(n=int((d_te == s).sum()), accuracy=float((P[d_te == s] == y_te[d_te == s]).mean())) for s in sorted(set(d_te.tolist()))}
# inference timing (batch of 1, this device)
xb = tens(x_te[:1]).to(DEVICE);
with torch.no_grad():
    for _ in range(10): model(xb)
    t = time.perf_counter();
    for _ in range(100): model(xb)
    ms = (time.perf_counter() - t) * 10
print(f"TEST accuracy {test['accuracy']:.4f}  macro-F1 {test['macro_f1']:.4f}  (n={len(y_te)})  | val acc {val['accuracy']:.4f}")
print("per source set:", by_src); print(f"inference {ms:.2f} ms/image on {DEVICE}")
json.dump(dict(run_id=RUN_ID, status="completed", task="isolated Bangla digit classification", test=test, val=dict(accuracy=val["accuracy"], macro_f1=val["macro_f1"]), by_source_set_test=by_src,
               inference_ms_per_image=dict(value=ms, device=DEVICE, batch=1), epochs=EPOCHS, n_train=int(len(x_tr)), n_val=int(len(x_va)), n_test=int(len(x_te)),
               note="Isolated handwritten digits from NumtaDB only. Not a khatian reader; says nothing about land-verification performance."), open(os.path.join(OUT, "metrics.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
json.dump(dict(labels=[str(k) for k in range(10)], labels_bn=list("০১২৩৪৫৬৭৮৯"), matrix=cm.tolist(), axis="rows=true class, cols=predicted"), open(os.path.join(OUT, "confusion_matrix.json"), "w", encoding="utf-8"), ensure_ascii=False)

In [ ]:
import importlib, subprocess, sys
for pkg in ("onnx", "onnxscript", "onnxruntime"):
    try: importlib.import_module(pkg)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
model.eval().cpu(); dummy = torch.zeros(1, 1, 32, 32)
onnx_path = os.path.join(OUT, "numta_cnn.onnx")
kw = dict(input_names=["image"], output_names=["logits"], dynamic_axes=dict(image={0: "b"}, logits={0: "b"}), opset_version=17)
try: torch.onnx.export(model, dummy, onnx_path, dynamo=False, **kw)
except TypeError: torch.onnx.export(model, dummy, onnx_path, **kw)
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path); xb = tens(x_te[:64])
a = model(xb).detach().numpy(); b = sess.run(None, dict(image=xb.numpy()))[0]
parity = float(np.abs(a - b).max()); agree = float((a.argmax(1) == b.argmax(1)).mean()); print("onnx parity: max|diff|", parity, "argmax agreement", agree)
json.dump(dict(input="image float32 [B,1,32,32], values 0..1, ink bright (255-gray)/255", output="logits [B,10] (apply softmax yourself; NOT calibrated confidence)", classes=list("০১২৩৪৫৬৭৮৯"), onnx_parity_max_abs_diff=parity), open(os.path.join(OUT, "preprocessing.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
open(os.path.join(OUT, "model_card.md"), "w", encoding="utf-8").write(f'''# Model card — {RUN_ID}
**Task.** Classify one isolated handwritten Bangla digit (০–৯) from a 32x32 grayscale image. Input assistance for reading single digits in form fields; not a document reader.
**Data.** NumtaDB (Bengali.AI, Kaggle `BengaliAI/numta`, CC BY-SA 4.0) training sets a–e, preprocessed to 32x32 inverted grayscale. Train {len(x_tr)}, validation {len(x_va)}, test {len(x_te)} images. Split is by image, seeded; writer identities are not available, so writer leakage between splits cannot be excluded (disclosed).
**Model.** Small CNN ({sum(p.numel() for p in model.parameters()) / 1e6:.2f} M params), AdamW + OneCycle, {EPOCHS} epochs, light affine/noise augmentation. Selected by validation accuracy; evaluated once on test.
**Results (test).** accuracy {test['accuracy']:.4f}, macro-F1 {test['macro_f1']:.4f}. Per-source-set accuracy and the confusion matrix are in metrics.json / confusion_matrix.json.
**Export.** ONNX opset 17, parity vs PyTorch max|diff| {parity:.2e}.
**Limitations.** Isolated digits only; no multi-digit segmentation; performance on digits cropped from real khatians is **not measured**. Softmax outputs are not calibrated probabilities.
''')
print(sorted(os.listdir(OUT)))